# Compare simulated water levels with observations

Compares D-Flow FM water levels from the coupled runs against measured water levels
at the tide and estuary gages inside the model domain, for each grid resolution.

## What this notebook found

Two systematic offsets between model and gage, both traced to the hydrodynamic
boundary rather than to the coupling or the grid. **Neither is a small correction,
and no skill number from this notebook should be quoted until they are resolved.**

1. **The model is about half a meter high.** The median model-minus-gage bias is
   +0.47 m and it is nearly the same at every station. The surge boundary series,
   `WaterLevel2010_surge.bc`, has a mean of +0.487 m over the same window. The two
   agree to 13 mm, so the offset is the boundary's own mean carried through the
   domain, not anything the model does to it. That series is behaving as a water
   level in some other datum rather than as a zero-mean surge.
2. **The model tide is about five hours early.** Cross-correlation peaks at a lag of
   245 to 450 minutes depending on station, clustered near 300--345. A common
   five-hour component is what a phase reference in local standard time would
   produce against a model running `Tzone 0`; the tidal boundary
   (`Waterlevel_lst.bc`) is astronomic, so its phases carry that reference. The
   spread about the common offset is real propagation difference.

Correlation against the raw series is therefore *negative* -- five hours is most of
a semidiurnal half-period. That is a reference error, not a model failure: once both
offsets are removed the model tracks the gages closely, and the adjusted statistics
below are the honest measure of tidal skill.

## Method

**One coupling interval is enough.** Water level at these gages is set by the tidal
boundary, not the sewer exchange: across the sweep from 30-minute to daily coupling
the coarse water levels differ by at most 0.07 mm, 0.0015 % of tidal range. Seven
coupling intervals would draw seven identical lines. Grid resolution is what varies
here.

**Sources**, both public, fetched once and cached under `data/GP/observations/` so
the notebook re-runs offline:

- USGS NWIS parameter **62620**, estuary water surface elevation above NAVD 1988, in
  feet. Eight gages.
- NOAA CO-OPS `water_level`, 6-minute verified, metric. Six tide stations; five on
  NAVD 1988. New Haven publishes no NAVD datum, so it is fetched on MSL and its bias
  is not comparable with the rest.

**Datums and time.** Everything is reduced to meters above NAVD 1988 and to UTC. The
`.mdu` sets `RefDate 20000101` and `Tzone 0`, and the history file carries
`seconds since 2000-01-01 00:00:00 +00:00`, so model output is already UTC.

**Coverage is not nested.** Stations that a grid reports as `dry`, `disconnected` or
`outside` are excluded for that grid rather than plotted as flat lines. The highres
grid resolves Battery, which coarse misses, but loses New London, which midres
resolves.

In [ ]:
%matplotlib inline
import pathlib as pl

import flopy.plot.styles as styles
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import xarray as xr

In [ ]:
COUPLING = "08.00H"
N_JUNCTIONS = 244
RESOLUTIONS = {"coarse": "gp_coarse", "midres": "gp_medium", "highres": "gp_high"}

SPINUP_DAYS = 5.0                       # cold start; not a fair test of the model
PLOT_START, PLOT_END = "2010-02-01", "2010-02-15"

# Wide enough to contain the true peak. The first version of this notebook searched
# +/- 3 h and railed at the bound at every station, reporting exactly +/-180 min --
# a clipped optimum, not a measurement. It must exceed half a semidiurnal period
# (6.21 h) or a real offset of this size cannot be seen at all.
MAX_LAG_H = 8.0

ROOT = pl.Path.cwd().parent
RESULTS = ROOT / "results" / "gp"
OBS_DIR = ROOT / "data" / "GP" / "observations"
OBS_DIR.mkdir(parents=True, exist_ok=True)
# Notebook-local, like the other step3 notebooks. Not docs/GP/figures: these are not
# manuscript figures while the two boundary offsets above are unresolved.
FIGS = pl.Path.cwd() / "figures"
FIGS.mkdir(exist_ok=True)

FT2M = 0.3048

In [ ]:
# Keyed by the station name D-Flow FM writes into the history file. CO-OPS ids come
# from matching the model observation-point coordinates against the CO-OPS station
# list; only matches within ~0.5 km are used. Eaton, New Rochelle, Port Jefferson,
# Silver Eel Pond, South Jamesport and Willets Point are model output locations
# named after places, not gages -- CO-OPS returns no 2010 data for any of them -- so
# they carry no observation here.
STATIONS = {
    "Battery":       ("coops", "8518750",  "NAVD"),
    "Bridgeport":    ("coops", "8467150",  "NAVD"),
    "Kings Point":   ("coops", "8516945",  "NAVD"),
    "Montauk":       ("coops", "8510560",  "NAVD"),
    "New London":    ("coops", "8461490",  "NAVD"),
    "New Haven":     ("coops", "8465705",  "MSL"),   # no NAVD datum published
    "usgs_01302250": ("nwis",  "01302250", "NAVD"),
    "usgs_01302845": ("nwis",  "01302845", "NAVD"),
    "usgs_01309225": ("nwis",  "01309225", "NAVD"),
    "usgs_01310521": ("nwis",  "01310521", "NAVD"),
    "usgs_01310740": ("nwis",  "01310740", "NAVD"),
    "usgs_01311145": ("nwis",  "01311145", "NAVD"),
    "usgs_01311850": ("nwis",  "01311850", "NAVD"),
    "usgs_01311875": ("nwis",  "01311875", "NAVD"),
}
LABEL = {k: (k.replace("usgs_", "USGS ") if k.startswith("usgs_") else k)
         for k in STATIONS}

In [ ]:
def _coops(station, datum, t0, t1):
    """CO-OPS caps water_level at 31 days per request, so fetch by month."""
    starts = pd.DatetimeIndex([pd.Timestamp(t0)]).union(
        pd.date_range(t0, t1, freq="MS"))
    frames = []
    for start in starts:
        stop = min(pd.Timestamp(t1),
                   start + pd.offsets.MonthEnd(1) + pd.Timedelta(days=1))
        r = requests.get(
            "https://api.tidesandcurrents.noaa.gov/api/prod/datagetter",
            params={"product": "water_level", "application": "usgs_liss",
                    "begin_date": start.strftime("%Y%m%d"),
                    "end_date": stop.strftime("%Y%m%d"), "datum": datum,
                    "station": station, "time_zone": "gmt", "units": "metric",
                    "format": "json"}, timeout=120)
        j = r.json()
        if "data" not in j:
            raise RuntimeError(f"CO-OPS {station}: {j.get('error', j)}")
        d = pd.DataFrame(j["data"])
        d = d[d["v"] != ""]
        frames.append(pd.DataFrame({"time": pd.to_datetime(d["t"]),
                                    "obs": d["v"].astype(float)}))
    return (pd.concat(frames).drop_duplicates("time")
            .set_index("time").sort_index())


def _nwis(site, t0, t1):
    """Parameter 62620 is water surface elevation above NAVD 1988, in FEET."""
    r = requests.get("https://waterservices.usgs.gov/nwis/iv/",
                     params={"format": "json", "sites": site,
                             "parameterCd": "62620",
                             "startDT": pd.Timestamp(t0).strftime("%Y-%m-%d"),
                             "endDT": pd.Timestamp(t1).strftime("%Y-%m-%d"),
                             "siteStatus": "all"}, timeout=180)
    ts = r.json()["value"]["timeSeries"]
    if not ts:
        raise RuntimeError(f"NWIS {site}: no 62620 series in window")
    d = pd.DataFrame(ts[0]["values"][0]["value"])
    d = d[d["value"].astype(float) > -999999]
    # dateTime carries a UTC offset; normalise then drop the tz so the index matches
    # the model's naive-UTC times.
    t = pd.to_datetime(d["dateTime"], format="ISO8601", utc=True).dt.tz_localize(None)
    return (pd.DataFrame({"time": t, "obs": d["value"].astype(float) * FT2M})
            .drop_duplicates("time").set_index("time").sort_index())


def observations(name, t0, t1, refresh=False):
    """Cached to CSV so the notebook re-runs offline and the comparison is against
    a fixed snapshot rather than a live service."""
    src, sid, datum = STATIONS[name]
    cache = OBS_DIR / f"{sid}_{datum}.csv"
    if cache.is_file() and not refresh:
        return pd.read_csv(cache, index_col=0, parse_dates=True)
    d = _coops(sid, datum, t0, t1) if src == "coops" else _nwis(sid, t0, t1)
    d.to_csv(cache)
    return d

In [ ]:
# station_his.nc is written by notebooks-GP/extract_station_his.py, which pulls the
# station series out of the run directory -- where a rerun would overwrite them --
# and tags each station with whether the grid resolves it.
model = {}
for res, prefix in RESOLUTIONS.items():
    p = RESULTS / f"{prefix}_{COUPLING}_n{N_JUNCTIONS}" / "station_his.nc"
    if not p.is_file():
        print(f"MISSING {res}: {p.relative_to(ROOT)}")
        continue
    ds = xr.open_dataset(p)
    names = [str(s) for s in ds["station_name"].values]
    model[res] = {
        "wl": pd.DataFrame(ds["waterlevel"].values,
                           index=pd.DatetimeIndex(ds["time"].values), columns=names),
        "status": dict(zip(names, (str(s) for s in ds["status"].values))),
    }
    n_ok = sum(v == "ok" for v in model[res]["status"].values())
    print(f"{res:8s} {len(model[res]['wl']):,} times  {n_ok}/{len(names)} stations ok")

T0 = min(m["wl"].index[0] for m in model.values())
T1 = max(m["wl"].index[-1] for m in model.values())
SCORE_FROM = T0 + pd.Timedelta(days=SPINUP_DAYS)
print(f"\nwindow {T0:%Y-%m-%d %H:%M} to {T1:%Y-%m-%d %H:%M} UTC, "
      f"scored from {SCORE_FROM:%Y-%m-%d}")

In [ ]:
obs = {}
for name in STATIONS:
    try:
        obs[name] = observations(name, T0, T1 + pd.Timedelta(days=1))
        src, sid, datum = STATIONS[name]
        print(f"{LABEL[name]:16s} {src:5s} {sid:9s} {datum:4s} n={len(obs[name]):,}")
    except Exception as e:
        print(f"{LABEL[name]:16s} FAILED {type(e).__name__}: {e}")

### Scoring

Observations are interpolated onto the model times, so every resolution is scored on
the same instants. Interpolation is limited to 30 minutes, leaving a gap in the
record as a gap rather than bridging it with a straight line across tidal cycles.

Two sets of numbers are reported:

- **raw** -- model against gage as they stand. Dominated by the two reference
  offsets, so `r` is negative and RMSE is around a meter. These are the numbers that
  reveal the offsets.
- **adjusted** -- the model shifted by its own best-fit lag and its mean bias
  removed. This is what the model does with the tide once the reference errors are
  taken out, and it is the only skill measure here worth reading.

Presenting only the adjusted numbers would hide two real setup problems; presenting
only the raw ones would say the model has no skill, which is also untrue.

In [ ]:
def align(mod, ob):
    """Observations onto model times, gaps preserved."""
    both = ob["obs"].reindex(ob.index.union(mod.index)).interpolate(
        method="time", limit=6, limit_area="inside").reindex(mod.index)
    return pd.DataFrame({"mod": mod, "obs": both}).dropna()


def best_lag(d, dt_min):
    """Cross-correlation peak within +/- MAX_LAG_H, returning the aligned pair.

    The bound must exceed half a semidiurnal period or a large offset is invisible;
    it must stay under a full period or the peak becomes ambiguous."""
    n = int(MAX_LAG_H * 60 / dt_min)
    m = d["mod"].values
    o = d["obs"].values
    best_r, best_k = -np.inf, 0
    for k in range(-n, n + 1):
        a, b = (m[-k:], o[:len(o) + k]) if k < 0 else (
            (m[:-k], o[k:]) if k > 0 else (m, o))
        if a.size < 500:
            continue
        r = float(np.corrcoef(a - a.mean(), b - b.mean())[0, 1])
        if r > best_r:
            best_r, best_k = r, k
    k = best_k
    a, b = (m[-k:], o[:len(o) + k]) if k < 0 else (
        (m[:-k], o[k:]) if k > 0 else (m, o))
    return best_k * dt_min, best_r, a, b


rows = []
for res, mdl in model.items():
    wl = mdl["wl"].loc[SCORE_FROM:]
    dt_min = (wl.index[1] - wl.index[0]).total_seconds() / 60.0
    for name in STATIONS:
        if name not in obs or name not in wl.columns:
            continue
        st = mdl["status"].get(name, "?")
        base = dict(resolution=res, station=LABEL[name], status=st)
        if st != "ok":
            rows.append(base | dict(n=0))
            continue
        d = align(wl[name], obs[name])
        if len(d) < 500:
            rows.append(base | dict(status="short", n=len(d)))
            continue
        err = d["mod"] - d["obs"]
        lag, r_at_lag, a, b = best_lag(d, dt_min)
        ea = (a - a.mean()) - (b - b.mean())          # lag- and bias-corrected
        rows.append(base | dict(
            n=len(d),
            bias=err.mean(),
            rmse_raw=float(np.sqrt((err ** 2).mean())),
            r_raw=float(d["mod"].corr(d["obs"])),
            lag=lag,
            rmse_adj=float(np.sqrt((ea ** 2).mean())),
            r_adj=r_at_lag,
            amp=float(a.std() / b.std())))

stats = pd.DataFrame(rows)
print(f"{len(stats)} station-resolution pairs")

In [ ]:
ok = stats[stats["status"] == "ok"]
cols = [c for c in RESOLUTIONS if c in stats["resolution"].unique()]

piv = ok.pivot(index="station", columns="resolution")
print("Adjusted skill -- model shifted by its best-fit lag, mean bias removed\n")
print(pd.concat([piv["rmse_adj"][cols].add_suffix(" RMSE"),
                 piv["amp"][cols].add_suffix(" amp"),
                 piv["r_adj"][cols].add_suffix(" r")],
                axis=1).round(3).to_string(na_rep="  --"))

print("\n\nThe two offsets, which the adjustment removes\n")
print(pd.concat([piv["bias"][cols].add_suffix(" bias_m"),
                 piv["lag"][cols].add_suffix(" lag_min")],
                axis=1).round(3).to_string(na_rep="  --"))

print("\n\nMedian over stations resolved by each grid")
print(ok.groupby("resolution")[["bias", "lag", "rmse_raw", "r_raw",
                                "rmse_adj", "r_adj", "amp"]]
      .median().round(3).to_string())

ex = stats[stats["status"] != "ok"]
print("\n\nExcluded (grid does not resolve the station)")
print(ex.pivot(index="station", columns="resolution", values="status")
      .to_string(na_rep="  --") if len(ex) else "  none")

### Tracing the offsets to the boundary

The bias is checked against the boundary forcing directly rather than being left as
an unexplained model-gage difference.

In [ ]:
# WaterLevel2010_surge.bc is a daily time series added to the astronomic tide on the
# same boundary. If it carried only surge its mean would be near zero.
bc = (ROOT / "dflow-fm" / "coarse" / f"run_gp_coarse_{COUPLING}_n{N_JUNCTIONS}"
      / "WaterLevel2010_surge.bc")
t, v = [], []
for line in bc.read_text().splitlines():
    s = line.split()
    if len(s) == 2:
        try:
            t.append(float(s[0]))
            v.append(float(s[1]))
        except ValueError:
            pass
tt = pd.to_datetime("2000-01-01") + pd.to_timedelta(t, unit="s")
sel = (tt >= SCORE_FROM) & (tt <= T1)
surge_mean = float(np.array(v)[sel].mean())

med_bias = float(ok["bias"].median())
print(f"surge boundary mean over the scored window : {surge_mean:+.3f} m")
print(f"median model-minus-gage bias               : {med_bias:+.3f} m")
print(f"unexplained                                : {med_bias - surge_mean:+.3f} m")
print("\nThe boundary series accounts for the offset almost exactly, so it is a")
print("datum/reference problem in the forcing, not something the model introduces.")

print(f"\nlag by station: median {ok['lag'].median():.0f} min, "
      f"range {ok['lag'].min():.0f} to {ok['lag'].max():.0f} min")
print("A five-hour common component is what an astronomic phase reference in local")
print("standard time gives against a model running Tzone 0; the spread about it is")
print("propagation across the domain.")

In [ ]:
# Time series, raw and corrected. Both are shown: the corrected panel alone would
# conceal how far apart the raw series are.
common = sorted(s for s in ok["station"].unique()
                if (stats[stats["station"] == s]["status"] == "ok").all())
sel_st = common[:4]
COLOR = {"coarse": "#d62728", "midres": "#1f77b4", "highres": "#2ca02c"}
inv = {v: k for k, v in LABEL.items()}
print("panels:", ", ".join(sel_st))

with styles.USGSPlot():
    fig, axs = plt.subplots(nrows=len(sel_st), ncols=2, figsize=(7.5, 1.5 * len(sel_st)),
                            layout="constrained", sharex=True)
    for row, lab in enumerate(sel_st):
        name = inv[lab]
        o = obs[name].loc[PLOT_START:PLOT_END]
        for col in (0, 1):
            ax = axs[row, col]
            ax.plot(o.index, o["obs"], color="0.25", lw=1.3, label="Observed", zorder=2)
            for res in cols:
                if model[res]["status"].get(name) != "ok":
                    continue
                s = model[res]["wl"][name].loc[PLOT_START:PLOT_END]
                y, idx = s.values, s.index
                if col == 1:
                    row_st = ok[(ok["station"] == lab) & (ok["resolution"] == res)]
                    if row_st.empty:
                        continue
                    idx = idx + pd.Timedelta(minutes=float(row_st["lag"].iloc[0]))
                    y = y - float(row_st["bias"].iloc[0])
                ax.plot(idx, y, color=COLOR[res], lw=0.8, label=res, zorder=3)
            ax.tick_params(labelsize=7, top=False)
            if col == 0:
                ax.set_ylabel("m", fontsize=7)
            styles.heading(ax=ax, heading=f"{lab}" + ("" if col == 0 else ", corrected"),
                           fontsize=7.5)
    h, l = axs[0, 0].get_legend_handles_labels()
    styles.graph_legend(ax=axs[-1, 0], handles=h, labels=l, loc="lower center",
                        bbox_to_anchor=(1.05, -0.95), ncol=4, frameon=False, fontsize=7)
    for c in (0, 1):
        styles.xlabel(ax=axs[-1, c], label="Date, 2010 (UTC)")
    fig.savefig(FIGS / "obs_timeseries.pdf", bbox_inches="tight")

In [ ]:
# Adjusted error by station and resolution.
with styles.USGSPlot():
    fig, axs = plt.subplots(ncols=3, figsize=(7.5, 3.0), layout="constrained")
    order = common
    for ax, (col, lab, ref) in zip(axs, [
            ("rmse_adj", "Adjusted RMSE, in meters", None),
            ("amp", "Amplitude ratio", 1.0),
            ("lag", "Lag, in minutes", 0.0)]):
        w = 0.26
        for i, res in enumerate(cols):
            sub = ok[ok["resolution"] == res].set_index("station").reindex(order)
            ax.barh(np.arange(len(order)) + (i - 1) * w, sub[col].values,
                    height=w, color=COLOR[res], label=res)
        if ref is not None:
            ax.axvline(ref, color="0.35", lw=0.8, linestyle=(0, (3, 2)), zorder=1)
        ax.set_yticks(np.arange(len(order)))
        ax.set_yticklabels(order if col == "rmse_adj" else [], fontsize=7)
        ax.tick_params(labelsize=7, top=False)
        styles.xlabel(ax=ax, label=lab)
    h, l = axs[0].get_legend_handles_labels()
    styles.graph_legend(ax=axs[1], handles=h, labels=l, loc="lower center",
                        bbox_to_anchor=(0.5, -0.30), ncol=3, frameon=False, fontsize=7)
    fig.savefig(FIGS / "obs_error_summary.pdf", bbox_inches="tight")

### What to do next

The two offsets are in the hydrodynamic boundary and are independent of the
coupling, so they do not affect the sewer-tracer results, which are internal to the
model. They do have to be resolved before any water-level comparison goes into the
manuscript:

1. Establish the datum and the intent of `WaterLevel2010_surge.bc`. Its mean of
   +0.487 m is the whole of the model's high bias. If it is a total water level
   rather than a surge anomaly, the astronomic tide is being added to something that
   already contains a mean sea level.
2. Establish the time reference of the astronomic phases in `Waterlevel_lst.bc`
   against `Tzone 0`. A five-hour shift is exactly EST.

Neither can be settled from the run directory alone -- both need whoever built the
boundary. Until then the adjusted statistics stand as a measure of tidal skill, and
the raw ones should not be quoted at all.